# Ranking województw ze względu na potencjał rolnictwa

Przedmiotem niniejszego projektu jest ocena i przestrzenna klasyfikacja polskich województw pod kątem ich potencjału rolniczego. Temat oraz metodyka badania zostały zainspirowane publikacją pt. „Ranking województw ze względu na potencjał rolnictwa” (J. Kisielińska, 2017).

Zgodnie z informacjami przedstawionymi w ww. publikacji, potencjał rolniczy jest zjawiskiem złożonym, charakteryzującym się silnym zróżnicowaniem regionalnym w skali kraju. Z punktu widzenia analizy danych, niemożliwe jest wyrażenie go za pomocą pojedynczego wskaźnika, z czego wynika potrzeba stosowania analizy wielowymiarowej.

Głównym celem projektu jest pomiar potencjału rolniczego poszczególnych województw w oparciu o aktualne dane statystyczne (z roku 2023) oraz zestawienie nowo powstałego układu z wynikami historycznymi przedstawionymi w publikacji.

## Wybór zmiennych diagnostycznych

Do analizy wybrano zestaw ośmiu zmiennych diagnostycznych wykorzystanych w publikacji referencyjnej. Są to:
   - $x_1$: udział użytków rolnych w powierzchni województwa
   - $x_2$: udział gruntów ugorowanych
   - $x_3$: udział zmeliorowanych użytków rolnych
   - $x_4$: pracujący w rolnictwie na 100 ha użytków rolnych
   - $x_5$: nakłady inwestycyjne w rolnictwie i łowiectwie na 100 ha użytków rolnych w tys. zł
   - $x_6$: wartość brutto środków trwałych w rolnictwie i łowiectwie na 100 ha użytków rolnych w tys. zł
   - $x_7$: zużycie nawozów mineralnych lub chemicznych na 1 ha użytków rolnych w kg
   - $x_8$: globalna produkcja rolnicza na 1 ha użytków rolnych w tys. zł

Dane zostały zaczerpnięte z "Rocznika Statystycznego Rolnictwa 2025" [GUS 2026] oraz przeliczone na jednostki z publikacji.

Zmienne $x_1, x_3, x_5, x_6, x_7, x_8$ uznano za stymulanty, a zmienne  $x_2, x_4$ za destymulanty.

In [1]:
import pandas as pd
from IPython import display

df = pd.read_csv("data.csv", sep=";", decimal=",")
df = pd.DataFrame(df)
stimulants = ["x1", "x3", "x5", "x6", "x7", "x8"]
destimulants = ["x2", "x4"]
all_vars = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8"]

## Wstępna analiza danych

## Porządkowanie liniowe

Ranking województw ze względu na potencjał rolnictwa dla 2023 roku sporządzono wybranymi metodami, oznaczając je następująco:
* metoda bezwzorcowa, normalizacja cech poprzez standaryzację (R1),
* metoda bezwzorcowa, normalizacja cech metodą unitaryzacji zerowanej (R2),
* wskaźnik rozwoju Hellwiga, normalizacja cech poprzez standaryzację (R5),
* metoda TOPSIS, normalizacja cech poprzez standaryzację (R6).

In [2]:
import linear_ordering
z_std, z_unit = linear_ordering.normalize_data(df, stimulants, all_vars)

df_res = linear_ordering.calculate_synthetic_variables(df, z_std, z_unit)

df_u, df_u_bar, best_ranking_code = linear_ordering.evaluate_similarity(df_res, len(df))

df_res["Klasa"] = linear_ordering.assign_classes(df_res[f"q_{best_ranking_code}"])

ranks_historical = {
    "Opolskie": 1,
    "Śląskie": 2,
    "Wielkopolskie": 3,
    "Kujawsko-pomorskie": 4,
    "Łódzkie": 5,
    "Dolnośląskie": 6,
    "Pomorskie": 7,
    "Mazowieckie": 8,
    "Lubelskie": 9,
    "Świętokrzyskie": 10,
    "Małopolskie": 11,
    "Zachodniopomorskie": 12,
    "Podlaskie": 13,
    "Warmińsko-mazurskie": 14,
    "Lubuskie": 15,
    "Podkarpackie": 16,
}

df_comp = linear_ordering.compare_with_historical(df_res, best_ranking_code, ranks_historical)

display(df_res[["Województwo", "R1", "R2", "R5", "R6", "Klasa"]])

TypeError: 'module' object is not callable

Tabela przedstawiająca pozycje poszczególnych województw ze względu na potencjał rolnictwa potwierdza spostrzeżenie wskazane w publikacji o możliwości wystąpienia różnic w pozycjach w rankingach uzyskanych różnymi metodami oraz konieczności stosowania miary określonej formułą pozwalającej wybrać wynik najbardziej podobny do pozostałych. Do wyróżnienia klas wykorzystano średnią i odchylenie standardowe zmiennej syntetycznej w następujący sposób:
* klasa I (bardzo wysoki poziom badanego zjawiska): $q_i \ge \bar{q} + 0,8416 \cdot s_q$
* klasa II (wysoki poziom): $\bar{q} + 0,8416 \cdot s_q > q_i \ge \bar{q} + 0,2533 \cdot s_q$
* klasa III (średni poziom): $\bar{q} + 0,2533 \cdot s_q > q_i \ge \bar{q} - 0,2533 \cdot s_q$
* klasa IV (niski poziom): $\bar{q} - 0,2533 \cdot s_q > q_i \ge \bar{q} - 0,8416 \cdot s_q$
* klasa V (bardzo niski poziom): $\bar{q} - 0,8416 \cdot s_q > q_i$

Do klasy I zakwalifikowały się województwa: Opolskie, Wielkopolskie i Kujawsko-Pomorskie. Są to regiony charakteryzujące się najwyższym natężeniem cech stymulujących, takich jak wysokie nakłady inwestycyjne, wysoka wartość środków trwałych czy wysoka produkcja globalna. Województwa te mają taką samą klasę we wszystkich badanych metodach porządkowania, co potwierdza ich wysoki potencjał rolniczy.

W klasie II znalazły się województwa Dolnośląskie oraz Podlaskie. Mają one poziom potencjału rolniczego powyżej średniej krajowej. Posiadają one korzystną strukturę zmiennych diagnostycznych, ale ustępują województwom w I klasie ze względu na nieznacznie gorsze wyniki w niektórych obszarach.

Klasa III jest najliczniejszą grupą, obejmującą 6 województw: Mazowieckie, Łódzkie, Pomorskie, Warmińsko-mazurskie, Śląskie oraz Lubelskie. Województwa te charakteryzują się przeciętnym poziomem potencjału rolniczego. Wartości ich miar syntetycznych oscylują wokół średniej arytmetycznej dla całego kraju, co sugeruje umiarkowane nasycenie czynnikami produkcji rolniczej.

Na podstawie analizy rozkładu zmiennej syntetycznej nie wyodrębniono żadnego województwa, które zakwalifikowałoby się do klasy IV (poziom niski).

Do klasy V zaliczono 5 województw: Zachodniopomorskie, Małopolskie, Świętokrzyskie, Lubuskie oraz Podkarpackie. Regiony te charakteryzują się zdecydowanie najniższymi wartościami miary syntetycznej we wszystkich analizowanych wariantach. Silnie oddziałują na nie zmienne o charakterze destymulant, przy jednoczesnym bardzo niskim udziale cech stymulujących (np. niska produkcja na 1 ha, niski udział użytków rolnych, czy w przypadku woj. podkarpackiego i małopolskiego – niekorzystna struktura obszarowa gospodarstw). Województwo Podkarpackie znajduje się na końcu zestawienia w 3 z 4 zastosowanych metod.

In [ ]:
display(linear_ordering.display_similarity(df_u_bar))

Otrzymane rankingi należy uznać za podobne o czym świaczą miary podobieństwa powyżej 0.9, tak samo jak było w badaniu opartym na danych z 2014 roku.

In [ ]:

display(df_comp)
linear_ordering.plot_ranking_comparison(df_comp)

Porównanie wyników własnego badania z ustaleniami dla roku 2014 pozwala na ocenę stabilności oraz dynamiki potencjału rolniczego województw.  Na wykresie kolorem zielonym oznaczono awans, czerwonym spadek, a fioletowym brak zmiany.

Tylko cztery województwa utrzymały swoje pozycje z 2014 roku. Na szczycie rankingu bez zmian pozostało wyłącznie województwo opolskie (1. miejsce). W dolnej części zestawienia swoje pozycje zachowały województwa: zachodniopomorskie (12. miejsce), lubuskie (15. miejsce) oraz podkarpackie (16. miejsce).
Największą zmianę dodatnią odnotowało województwo podlaskie, przesuwając się z 13. na 5. pozycję (awans o 8 miejsc). Drugi co do wielkości wzrost dotyczy województwa warmińsko-mazurskiego, które zmieniło pozycję z 14. na 9. (awans o 5 miejsc). Poprawę odnotowały również: dolnośląskie (z 6. na 4.), mazowieckie (z 8. na 6.), kujawsko-pomorskie (z 4. na 3.) oraz wielkopolskie (z 3. na 2.). Największa zmiana ujemna dotyczy województwa śląskiego, które spadło z 2. miejsca w 2014 roku na 10. pozycję w 2023 roku (spadek o 8 miejsc). Województwo świętokrzyskie zanotowało spadek o 4 pozycje, przechodząc z 10. na 14. miejsce. Spadki o 2 pozycje dotyczyły województw: łódzkiego, lubelskiego i małopolskiego, a o 1 pozycję – województwa pomorskiego. Oznacza to, że w nowszym badaniu zmiany były dwukierunkowe: sześć województw awansowało, sześć spadło, a cztery utrzymały pozycję. Nie potwierdza to wprost obserwowanego wcześniej wyrównywania różnic między województwami.


## Analiza skupień

In [ ]:
import clustering
x, names, df_indexed = clustering.prepare_clustering_data(df, z_std)
clustering.plot_dendrogram(x, names, k=3)

In [ ]:
df_heights = clustering.plot_merge_heights(Z)
df_heights

In [ ]:
df_k = clustering.select_k_hierarchical(z, x, k_range=range(2, 9))
clustering.plot_cluster_indices(df_k)
df_k

In [ ]:
df_ksumm, best_k = clustering.summarize_best_k(df_k)
df_ksumm

In [ ]:
from scipy.cluster.hierarchy import cut_tree
labels_h = cut_tree(Z, n_clusters=[best_k]).ravel()
print("Rozkład liczebności grup:",
      pd.Series(labels_h).value_counts().to_dict())

clustering.profile_groups(df, labels_h, ["x1","x2","x3","x4","x5","x6","x7","x8"])